In [2]:

import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("No GPU detected. Enable a GPU runtime before continuing.")

print('Hello')

Python version: 3.13.15
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
Hello


In [3]:

%pip install -q -U transformers datasets trl peft accelerate bitsandbytes
%pip uninstall -y torchao


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
import transformers
import datasets
import trl
import peft
import bitsandbytes as bnb
import accelerate

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("4-bit quantization class:", bnb.nn.Linear4bit)

Transformers: 5.20.2
Datasets: 5.1.0
TRL: 1.16.0
PEFT: 0.21.2
Accelerate: 1.15.0
bitsandbytes: 0.50.2
CUDA available: True
GPU: Tesla T4
4-bit quantization class: <class 'bitsandbytes.nn.modules.Linear4bit'>


In [6]:

from datasets import load_dataset

dataset = load_dataset("glaiveai/glaive-function-calling-v2", split="train")

print("Number of examples:", len(dataset))
print("Column names:", dataset.column_names)
print("\nFirst example:")
print(dataset[0])


README.md:   0%|          | 0.00/106 [00:00<?, ?B/s]

glaive-function-calling-v2.json: reconstructing file:   0%|          |  0.00B /  271MB            

glaive-function-calling-v2.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112960 [00:00<?, ? examples/s]

Number of examples: 112960
Column names: ['system', 'chat']

First example:
{'system': 'SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -\n{\n    "name": "get_exchange_rate",\n    "description": "Get the exchange rate between two currencies",\n    "parameters": {\n        "type": "object",\n        "properties": {\n            "base_currency": {\n                "type": "string",\n                "description": "The currency to convert from"\n            },\n            "target_currency": {\n                "type": "string",\n                "description": "The currency to convert to"\n            }\n        },\n        "required": [\n            "base_currency",\n            "target_currency"\n        ]\n    }\n}\n', 'chat': "USER: Can you book a flight for me from New York to London?\n\n\nASSISTANT: I'm sorry, but I don't have the capability to book flights. My current function allows me to get the exchange rate between two currencies.

In [7]:
print(dataset[0]['system'])

SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "get_exchange_rate",
    "description": "Get the exchange rate between two currencies",
    "parameters": {
        "type": "object",
        "properties": {
            "base_currency": {
                "type": "string",
                "description": "The currency to convert from"
            },
            "target_currency": {
                "type": "string",
                "description": "The currency to convert to"
            }
        },
        "required": [
            "base_currency",
            "target_currency"
        ]
    }
}



In [8]:
print(dataset[0]['chat'])

USER: Can you book a flight for me from New York to London?


ASSISTANT: I'm sorry, but I don't have the capability to book flights. My current function allows me to get the exchange rate between two currencies. If you need help with that, feel free to ask! <|endoftext|>





In [9]:
import re

# Filter categories more precisely to find true "no-tool" examples
patterns = {
    "Function-call examples": r"<functioncall>",
    "Tool-response examples": r"FUNCTION RESPONSE:",
    "No-tool / normal responses": r"^(?!.*<functioncall>)(?!.*FUNCTION RESPONSE:).*ASSISTANT:"
}

for label, pattern in patterns.items():
    # Using re.DOTALL and re.IGNORECASE to ensure multi-line matches work correctly
    matches = [
        i for i, row in enumerate(dataset)
        if re.search(pattern, row["chat"], re.IGNORECASE | re.DOTALL)
    ]
    print(f"\n{label}: {len(matches)}")
    if matches:
        # Let's print the first match to see the simple non-tool response
        i = matches[2]
        print(f"Example index: {i}")
        print("SYSTEM:", dataset[i]["system"][:700])
        print("CHAT:", dataset[i]["chat"][:1800])


Function-call examples: 63218
Example index: 3
SYSTEM: SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "generate_password",
    "description": "Generate a random password with specified criteria",
    "parameters": {
        "type": "object",
        "properties": {
            "length": {
                "type": "integer",
                "description": "The length of the password"
            },
            "include_numbers": {
                "type": "boolean",
                "description": "Include numbers in the password"
            },
            "include_special_characters": {
                "type": "boolean",
                "description": "Include special characters in 
CHAT: USER: I need a new password. Can you generate one for me?


ASSISTANT: Of course! How long would you like your password to be? And do you want it to include numbers and special characters? <|endoftext|>


USER: I want it to be 12 charact

In [11]:

import json
import re
from collections import Counter

tool_names = Counter()
valid_calls = 0
invalid_calls = 0
call_pattern = re.compile(r"<functioncall>\s*(\{.*?\})\s*<\|endoftext\|>", re.DOTALL)

for row in dataset:
    for match in call_pattern.finditer(row["chat"]):
        try:
            call = json.loads(match.group(1))
            if isinstance(call.get("name"), str) and "arguments" in call:
                tool_names[call["name"]] += 1
                valid_calls += 1
            else:
                invalid_calls += 1
        except (json.JSONDecodeError, AttributeError):
            invalid_calls += 1

print("Valid function calls:", valid_calls)
print("Invalid/unparsed calls:", invalid_calls)
print("Unique tool names:", len(tool_names))
print("\nMost common tools:")
for name, count in tool_names.most_common(15):
    print(f"{name}: {count}")


Valid function calls: 2910
Invalid/unparsed calls: 80918
Unique tool names: 109

Most common tools:
generate_random_quote: 515
get_random_fact: 439
get_random_joke: 367
generate_random_color: 297
get_random_quote: 219
get_current_time: 155
generate_random_fact: 112
get_news_headlines: 98
generate_uuid: 84
get_news: 80
calculate_bmi: 72
generate_quote: 44
generate_random_name: 34
get_quote_of_the_day: 34
generate_random_joke: 20


In [12]:

import re

indices = [
    i for i, row in enumerate(dataset)
    if "<functioncall>" in row["chat"].lower()
]

print("Rows containing functioncall:", len(indices))

for i in indices[:5]:
    chat = dataset[i]["chat"]
    start = chat.lower().find("<functioncall>")
    print(f"\n--- Dataset row {i} ---")
    print(repr(chat[max(0, start - 100):start + 700]))


Rows containing functioncall: 63218

--- Dataset row 1 ---
'USER: Can you tell me the latest news headlines for the United States?\n\n\nASSISTANT: <functioncall> {"name": "get_news_headlines", "arguments": \'{"country": "United States"}\'} <|endoftext|>\n\n\nFUNCTION RESPONSE: {"headlines": ["Biden announces new vaccine mandates", "Hurricane Ida devastates Louisiana", "Apple unveils new iPhone", "NASA\'s Perseverance rover collects first Mars rock sample"]}\n\n\nASSISTANT: Here are the latest news headlines for the United States:\n1. Biden announces new vaccine mandates\n2. Hurricane Ida devastates Louisiana\n3. Apple unveils new iPhone\n4. NASA\'s Perseverance rover collects first Mars rock sample <|endoftext|>\n\n\nUSER: That\'s interesting. What about the news in France?\n\n\nASSISTANT: <functioncall> {"name": "get_news_headlines", "arguments": \'{"count'

--- Dataset row 2 ---
'xt|>\n\n\nUSER: I would like it to be 12 characters long and yes, please include symbols.\n\n\nASSISTANT:

In [14]:

import json
import re
from collections import Counter

pattern = re.compile(
    r'<functioncall>\s*\{\s*"name"\s*:\s*"([^"]+)"\s*,\s*'
    r'"arguments"\s*:\s*\'(.*?)\'\s*\}',
    re.DOTALL
)

tool_names = Counter()
valid_calls = 0
invalid_calls = 0

for row in dataset:
    for match in pattern.finditer(row["chat"]):
        try:
            name = match.group(1)
            arguments = json.loads(match.group(2))

            if isinstance(arguments, dict):
                tool_names[name] += 1
                valid_calls += 1
            else:
                invalid_calls += 1
        except (json.JSONDecodeError, TypeError):
            invalid_calls += 1

print("Valid function calls:", valid_calls)
print("Invalid/unparsed calls:", invalid_calls)
print("Unique tool names:", len(tool_names))
print("\nTop 10 tools:")
for name, count in tool_names.most_common(10):
    print(f"{name}: {count}")


Valid function calls: 80771
Invalid/unparsed calls: 135
Unique tool names: 933

Top 10 tools:
calculate_distance: 5063
convert_currency: 4682
get_stock_price: 3809
calculate_discount: 3278
calculate_bmi: 3170
calculate_tip: 3107
calculate_age: 3048
generate_random_number: 3003
calculate_area: 2878
get_movie_details: 2510


In [15]:

import json
import re

row = next(
    row for row in dataset
    if "<functioncall>" in row["chat"].lower()
    and pattern.search(row["chat"])
)

match = pattern.search(row["chat"])
name = match.group(1)
arguments = json.loads(match.group(2))

chat_prefix = row["chat"][:match.start()]
last_user = re.findall(
    r"USER:\s*(.*?)(?=\n\s*\n\s*(?:ASSISTANT|FUNCTION RESPONSE):|\Z)",
    chat_prefix,
    re.DOTALL
)

print("SYSTEM / TOOL DEFINITIONS:")
print(row["system"][:1500])

print("\nCONVERSATION CONTEXT:")
print(chat_prefix[-1200:])

print("\nTARGET TOOL NAME:")
print(name)

print("\nTARGET ARGUMENTS:")
print(json.dumps(arguments, indent=2))

print("\nPRECEDING USER MESSAGES:")
print(last_user[-2:])


SYSTEM / TOOL DEFINITIONS:
SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "get_news_headlines",
    "description": "Get the latest news headlines",
    "parameters": {
        "type": "object",
        "properties": {
            "country": {
                "type": "string",
                "description": "The country for which to fetch news"
            }
        },
        "required": [
            "country"
        ]
    }
}


CONVERSATION CONTEXT:
USER: Can you tell me the latest news headlines for the United States?


ASSISTANT: 

TARGET TOOL NAME:
get_news_headlines

TARGET ARGUMENTS:
{
  "country": "United States"
}

PRECEDING USER MESSAGES:
['Can you tell me the latest news headlines for the United States?']


## Cell 11: Build normalized function-calling examples
We'll convert the raw calls into consistent JSON targets. This makes it easier to train Qwen3 to predict both the correct tool name and its arguments.

In [16]:

import json
import re
from collections import Counter

call_examples = []
calls_per_conversation = Counter()

for row_idx, row in enumerate(dataset):
    chat = row["chat"]

    for match in pattern.finditer(chat):
        try:
            tool_name = match.group(1)
            arguments = json.loads(match.group(2))

            if not isinstance(arguments, dict):
                continue

            # Keep the conversation before this tool call as context.
            context = chat[:match.start()].strip()

            # Remove the unfinished assistant prefix, if present.
            context = re.sub(
                r"\s*ASSISTANT:\s*$",
                "",
                context,
                flags=re.IGNORECASE
            ).strip()

            if not context:
                continue

            target = json.dumps({
                "name": tool_name,
                "arguments": arguments
            }, ensure_ascii=False)

            call_examples.append({
                "source_idx": row_idx,
                "system": row["system"],
                "context": context,
                "target": target
            })

            calls_per_conversation[row_idx] += 1

        except (json.JSONDecodeError, TypeError):
            continue

print("Normalized training examples:", len(call_examples))
print("Conversations with tool calls:", len(calls_per_conversation))
print("Conversations with multiple calls:",
      sum(n > 1 for n in calls_per_conversation.values()))

print("\nExample input context:")
print(call_examples[0]["context"][-500:])

print("\nExample target:")
print(call_examples[0]["target"])


Normalized training examples: 80771
Conversations with tool calls: 60722
Conversations with multiple calls: 19049

Example input context:
USER: Can you tell me the latest news headlines for the United States?

Example target:
{"name": "get_news_headlines", "arguments": {"country": "United States"}}


## Cell 12: Split the dataset into train, validation, and test sets
We'll use a 80/10/10 split by conversation, not by individual function call. This prevents calls from the same conversation appearing in multiple splits, which could cause data leakage.

In [17]:

from sklearn.model_selection import train_test_split

# Split by original conversation index to prevent leakage.
conversation_ids = sorted(calls_per_conversation.keys())

train_ids, temp_ids = train_test_split(
    conversation_ids,
    test_size=0.20,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

train_examples = [
    ex for ex in call_examples if ex["source_idx"] in train_set
]
val_examples = [
    ex for ex in call_examples if ex["source_idx"] in val_set
]
test_examples = [
    ex for ex in call_examples if ex["source_idx"] in test_set
]

print("Train conversations:", len(train_set))
print("Validation conversations:", len(val_set))
print("Test conversations:", len(test_set))

print("\nTrain examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Test examples:", len(test_examples))

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

print("\nNo conversation leakage between splits.")


Train conversations: 48577
Validation conversations: 6072
Test conversations: 6073

Train examples: 64633
Validation examples: 8078
Test examples: 8060

No conversation leakage between splits.


## Cell 13: Load Qwen3-1.7B and inspect its chat template

In [18]:

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Chat template available:", tokenizer.chat_template is not None)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("Pad token:", tokenizer.pad_token)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2 + 2?"}
]

print("\nFormatted example:")
print(tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
))


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Chat template available: True
EOS token: <|im_end|>
EOS token ID: 151645
Pad token: <|endoftext|>

Formatted example:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant



In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

# MODEL_NAME = "Qwen/Qwen3-1.7B"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype="auto",
#     device_map="auto"
# )

# print("Model loaded successfully!")
# print("Device:", model.device)

In [ ]:
# messages = [
#     {"role": "system", "content": "You are a helpful assistant."}
# ]

# while True:
#     user_input = input("\nYou: ")
# #
#     if user_input.lower() in ["exit", "quit", "bye"]:
#         print("Chat ended!")
#         break

#     messages.append({
#         "role": "user",
#         "content": user_input
#     })

#     inputs = tokenizer.apply_chat_template(
#         messages,
#         tokenize=True,
#         add_generation_prompt=True,
#         return_tensors="pt",
#         return_dict=True,
#         enable_thinking=False
#     ).to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=256,
#             do_sample=True,
#             temperature=0.7,
#             top_p=0.8
#         )

#     # Extract only the newly generated tokens
#     new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

#     response = tokenizer.decode(
#         new_tokens,
#         skip_special_tokens=True
#     )

#     print("\nQwen:", response)

#     messages.append({
#         "role": "assistant",
#         "content": response
#     })

## Cell 14: Load the base model
We'll load it in 4-bit quantization to fit your Tesla T4 GPU's 16 GB VRAM class.

In [19]:

import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model loaded successfully!")
print("Model dtype:", next(model.parameters()).dtype)
print("Model device:", model.device)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully!
Model dtype: torch.float16
Model device: cuda:0
GPU memory allocated: 2.17 GB


In [ ]:

def chat_with_model():
    messages = [
        {
            "role": "system",
            "content": "You are a helpful, friendly assistant. Answer clearly and naturally."
        }
    ]

    print("Chat with Qwen3-1.7B! Type 'exit' to stop.\n")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in {"exit", "quit"}:
            print("Chat ended.")
            break

        if not user_input:
            continue

        messages.append({"role": "user", "content": user_input})

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=True,
                temperature=0.7,
                top_p=0.8,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode only newly generated tokens.
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        print(f"Qwen: {response}\n")

        messages.append({"role": "assistant", "content": response})


chat_with_model()


Chat with Qwen3-1.7B! Type 'exit' to stop.

You: exit
Chat ended.


## Cell 16: Format examples for LoRA fine-tuning
We'll convert the parsed examples into Qwen3's chat format. The prompt contains the tool definitions and conversation context; the completion contains the function-call JSON the model should learn to generate.

In [20]:

from datasets import Dataset

def format_example(example):
    system_text = example["system"].removeprefix("SYSTEM:").strip()

    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": example["context"]},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return {
        "prompt": prompt,
        "completion": example["target"] + tokenizer.eos_token,
    }

train_dataset = Dataset.from_list([
    format_example(ex) for ex in train_examples
])

val_dataset = Dataset.from_list([
    format_example(ex) for ex in val_examples
])

print("Training rows:", len(train_dataset))
print("Validation rows:", len(val_dataset))

print("\nPROMPT PREVIEW:")
print(train_dataset[0]["prompt"][-800:])

print("\nCOMPLETION PREVIEW:")
print(train_dataset[0]["completion"])


Training rows: 64633
Validation rows: 8078

PROMPT PREVIEW:
<|im_start|>system
You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "get_news_headlines",
    "description": "Get the latest news headlines",
    "parameters": {
        "type": "object",
        "properties": {
            "country": {
                "type": "string",
                "description": "The country for which to fetch news"
            }
        },
        "required": [
            "country"
        ]
    }
}<|im_end|>
<|im_start|>user
USER: Can you tell me the latest news headlines for the United States?<|im_end|>
<|im_start|>assistant
<think>

</think>



COMPLETION PREVIEW:
{"name": "get_news_headlines", "arguments": {"country": "United States"}}<|im_end|>


## Cell 17: Configure LoRA

In [21]:

from peft import LoraConfig, get_peft_model

# Disable KV caching during training.
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


## Cell 18: Configure supervised fine-tuning
We'll use TRL's SFTTrainer with a conservative batch size for your Tesla T4. This cell only configures the trainer; it does not start training yet.

In [22]:

from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./qwen3-function-calling-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_steps=10,
    # warmup_ratio=0.03,
    max_length=1024,
    completion_only_loss=True,
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("Trainer configured successfully.")
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Effective batch size:", 2 * 8)
print("Maximum sequence length:", 1024)

Adding EOS to train dataset:   0%|          | 0/64633 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/64633 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/64633 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/64633 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/64633 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/8078 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/8078 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/8078 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/8078 [00:00<?, ? examples/s]

Trainer configured successfully.
Training examples: 64633
Validation examples: 8078
Effective batch size: 16
Maximum sequence length: 1024


## Cell 19: Verify the training labels

In [23]:
print("Training examples after preprocessing:", len(trainer.train_dataset))
print("Validation examples after preprocessing:", len(trainer.eval_dataset))

sample = trainer.train_dataset[0]
labels = sample["labels"]

if hasattr(labels, "tolist"):
    labels = labels.tolist()

valid_labels = sum(token != -100 for token in labels)

print("Tokens contributing to loss in first example:", valid_labels)
print("Total tokens in first example:", len(labels))

assert valid_labels > 0, (
    "This example has no active labels. Check completion-only loss configuration."
)

print("\nLabel verification passed.")

Training examples after preprocessing: 64633
Validation examples after preprocessing: 8078
Tokens contributing to loss in first example: 20
Total tokens in first example: 151

Label verification passed.


## CELL 19a: Validation

In [24]:

import json
import re
import torch
from tqdm.auto import tqdm

BASELINE_N = 1000
baseline_examples = test_examples[:BASELINE_N]

def extract_json_object(text):
    # Try to parse a JSON object from the generated response.
    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", text):
        try:
            obj, _ = decoder.raw_decode(text[match.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue

    return None

def evaluate_examples(examples, model_to_test):
    results = []

    model_to_test.eval()

    for ex in tqdm(examples):
        prompt = format_example(ex)["prompt"]
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        ).to(model_to_test.device)

        with torch.inference_mode():
            output = model_to_test.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        generated = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()

        predicted = extract_json_object(generated)
        expected = json.loads(ex["target"])

        results.append({
            "expected_tool": expected["name"],
            "predicted_tool": (
                predicted.get("name") if predicted else None
            ),
            "tool_correct": (
                predicted is not None
                and predicted.get("name") == expected["name"]
            ),
            "arguments_exact": (
                predicted is not None
                and predicted.get("arguments") == expected["arguments"]
            ),
            "valid_json": predicted is not None,
            "generated": generated,
        })

    total = len(results)
    print(f"\nExamples evaluated: {total}")
    print(f"Valid JSON rate: {sum(r['valid_json'] for r in results) / total:.1%}")
    print(f"Tool-name accuracy: {sum(r['tool_correct'] for r in results) / total:.1%}")
    print(f"Exact argument accuracy: {sum(r['arguments_exact'] for r in results) / total:.1%}")

    print("\n--- First 3 predictions ---")
    for i, r in enumerate(results[:3], 1):
        print(f"\nExample {i}")
        print("Expected tool:", r["expected_tool"])
        print("Predicted tool:", r["predicted_tool"])
        print("Arguments exact:", r["arguments_exact"])
        print("Generated:", r["generated"][:500])

    return results




In [ ]:
# Disable the untrained LoRA adapter for a clean base-model baseline.
with model.disable_adapter():
    baseline_results = evaluate_examples(baseline_examples, model)

## Cell 20: Start LoRA fine-tuning
This will begin training for one epoch. With 64,633 examples and an effective batch size of 16, expect approximately 4,040 optimizer steps. The actual time depends on sequence lengths and GPU speed.

In [25]:
import torch

torch.cuda.empty_cache()

print("Starting FP16 LoRA fine-tuning...")
train_result = trainer.train()

print("\nTraining complete!")
print("Training metrics:")

print(train_result.metrics)

Starting FP16 LoRA fine-tuning...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,0.011695,0.007428,0.009938,0.997724,894763.000000
400,0.001710,0.006122,0.005109,0.997960,1800566.000000
600,0.007855,0.007105,0.006378,0.997790,2697879.000000
800,0.007218,0.005656,0.006600,0.998125,3587875.000000
1000,0.006385,0.005594,0.004735,0.998329,4484326.000000
1200,0.004110,0.005419,0.003785,0.998230,5374802.000000
1400,0.002275,0.004561,0.003994,0.998447,6266293.000000
1600,0.007042,0.004993,0.004603,0.998405,7155874.000000
1800,0.006609,0.004483,0.004502,0.998396,8046129.000000
2000,0.001024,0.004828,0.004000,0.998231,8950939.000000



Training complete!
Training metrics:
{'train_runtime': 35392.9583, 'train_samples_per_second': 1.826, 'train_steps_per_second': 0.114, 'total_flos': 1.8351170894831616e+17, 'train_loss': 0.0065853996216181186, 'epoch': 1.0}


In [26]:
# Post-training evaluation: LoRA adapter enabled
BASELINE_N = 1000
baseline_examples = test_examples[:BASELINE_N]

finetuned_results = evaluate_examples(
    baseline_examples,
    model
)

  0%|          | 0/1000 [00:00<?, ?it/s]


Examples evaluated: 1000
Valid JSON rate: 100.0%
Tool-name accuracy: 100.0%
Exact argument accuracy: 96.4%

--- First 3 predictions ---

Example 1
Expected tool: calculate_tip
Predicted tool: calculate_tip
Arguments exact: True
Generated: {"name": "calculate_tip", "arguments": {"bill_amount": 50, "tip_percentage": 20}}

Example 2
Expected tool: analyze_website
Predicted tool: analyze_website
Arguments exact: True
Generated: {"name": "analyze_website", "arguments": {"website_url": "www.mywebsite.com", "metrics": ["traffic", "bounce rate", "page load time"]}}

Example 3
Expected tool: get_stock_price
Predicted tool: get_stock_price
Arguments exact: True
Generated: {"name": "get_stock_price", "arguments": {"symbol": "AAPL"}}


In [28]:

import json
import torch

# 1. Define the tools available to your model.
# Edit these tools or add your own.
tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Name of the city"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"]
                }
            },
            "required": ["city"]
        }
    },
    {
        "name": "get_stock_price",
        "description": "Get the latest stock price for a company.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "Stock ticker, such as AAPL or MSFT"
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "search_web",
        "description": "Search the internet for information on a topic.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "What to search for"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "calculate_tip",
        "description": "Calculate a restaurant tip from a bill amount and percentage.",
        "parameters": {
            "type": "object",
            "properties": {
                "bill_amount": {"type": "number"},
                "tip_percentage": {"type": "number"}
            },
            "required": ["bill_amount", "tip_percentage"]
        }
    },
    {
        "name": "send_email",
        "description": "Prepare an email to send to a recipient.",
        "parameters": {
            "type": "object",
            "properties": {
                "recipient": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["recipient", "subject", "body"]
        }
    }
]

# 2. Build the system prompt containing tool descriptions and schemas.
system_prompt = f"""
You are an AI assistant that selects the best tool for a user's request.

Available tools and their parameter schemas:
{json.dumps(tools, indent=2)}

Choose the most appropriate tool and provide its arguments.
Use only the listed tool names and follow their parameter schemas.
Return ONLY a JSON object in this exact format:
{{"name": "tool_name", "arguments": {{}}}}
Do not include explanations or Markdown.
"""

# 3. Interactive testing loop.
model.eval()

print("Interactive Tool-Calling Tester")
print("Enter 'quit' to stop.\n")

while True:
    user_prompt = input("Your request: ").strip()

    if user_prompt.lower() in ["quit", "exit", "q"]:
        print("Testing finished.")
        break

    if not user_prompt:
        continue

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("\nModel output:")
    try:
        # Extract the JSON object from the response.
        predicted = extract_json_object(generated)

        if predicted is None:
            print("Could not parse a valid JSON object.")
            print("Raw response:", generated)
        else:
            print(json.dumps(predicted, indent=2))

            selected_tool = predicted.get("name")
            available_names = [t["name"] for t in tools]

            if selected_tool in available_names:
                print(f"\nSelected tool: {selected_tool}")
                print("Tool exists in available tools: YES")
            else:
                print(f"\nSelected tool: {selected_tool}")
                print("Tool exists in available tools: NO")

    except Exception as e:
        print("Error:", e)

    print("\n" + "-" * 55 + "\n")


Interactive Tool-Calling Tester
Enter 'quit' to stop.



Your request:  It's raining in Islamabad. What's the current weather in Celsius?



Model output:
{
  "name": "get_weather",
  "arguments": {
    "city": "Islamabad",
    "unit": "celsius"
  }
}

Selected tool: get_weather
Tool exists in available tools: YES

-------------------------------------------------------



Your request:  How much should I tip on a $85 bill if I want to leave 18%?



Model output:
{
  "name": "calculate_tip",
  "arguments": {
    "bill_amount": 85,
    "tip_percentage": 18
  }
}

Selected tool: calculate_tip
Tool exists in available tools: YES

-------------------------------------------------------



Your request:  Prepare an email to ahmad@example.com with subject Project Update and body The project is ready for review.



Model output:
{
  "name": "send_email",
  "arguments": {
    "recipient": "ahmad@example.com",
    "subject": "Project Update",
    "body": "The project is ready for review."
  }
}

Selected tool: send_email
Tool exists in available tools: YES

-------------------------------------------------------



Your request:  exit


Testing finished.


In [30]:
import os
import json

SAVE_DIR = "/kaggle/working/qwen3-function-calling-lora-final"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the fine-tuned LoRA adapter and tokenizer
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save post-training evaluation results
with open(f"{SAVE_DIR}/finetuned_evaluation.json", "w") as f:
    json.dump(finetuned_results, f, indent=2, default=str)

# Save training metrics
with open(f"{SAVE_DIR}/training_metrics.json", "w") as f:
    json.dump(train_result.metrics, f, indent=2, default=str)

print("Saved successfully!")
print("Folder:", SAVE_DIR)
print("Files:", os.listdir(SAVE_DIR))

Saved successfully!
Folder: /kaggle/working/qwen3-function-calling-lora-final
Files: ['chat_template.jinja', 'tokenizer_config.json', 'adapter_model.safetensors', 'finetuned_evaluation.json', 'adapter_config.json', 'tokenizer.json', 'training_metrics.json', 'README.md']
